# Step 9 — RAG Service Prototype

Notebook ini menggabungkan seluruh pipeline menjadi sebuah fungsi `answer_question()`.

Alur:

1. Terima pertanyaan.
2. Retrieve Top-K chunks.
3. Susun context.
4. Bangun prompt.
5. Kirim ke LLM (opsional).
6. Kembalikan jawaban beserta sumber.


In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
from sentence_transformers import SentenceTransformer

DB_PATH = Path("./data/linux_docs.db")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

MODEL_NAME = "BAAI/bge-small-en-v1.5"
embedder = SentenceTransformer(MODEL_NAME)
TOP_K = 5


In [ ]:
def retrieve(question: str, top_k: int = TOP_K):
    qvec = embedder.encode(question, normalize_embeddings=True).astype(np.float32)

    cursor.execute("""
    SELECT
        p.url,
        p.title,
        c.section,
        c.content,
        e.embedding
    FROM embeddings e
    JOIN chunks c ON e.chunk_id = c.id
    JOIN pages p ON c.page_id = p.id
    """)

    results = []

    for row in cursor.fetchall():
        vec = np.frombuffer(row["embedding"], dtype=np.float32)
        score = float(np.dot(qvec, vec))

        results.append({
            "score": score,
            "url": row["url"],
            "title": row["title"],
            "section": row["section"],
            "content": row["content"]
        })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]


In [ ]:
def build_prompt(question: str, contexts: list[dict]) -> str:
    context = "\n\n".join(
        f"[Source: {c['title']} | {c['section']}]\n{c['content']}"
        for c in contexts
    )

    return f"""You are a Linux documentation assistant.

Use ONLY the context below to answer.

If the answer cannot be found, say you don't know.

Context:
{context}

Question:
{question}

Answer:
"""


In [ ]:
def answer_question(question: str):
    contexts = retrieve(question)

    prompt = build_prompt(question, contexts)

    return {
        "question": question,
        "prompt": prompt,
        "sources": [
            {
                "title": c["title"],
                "section": c["section"],
                "url": c["url"],
                "score": round(c["score"], 4)
            }
            for c in contexts
        ]
    }


In [ ]:
question = "How do I partition the disk during Arch Linux installation?"

result = answer_question(question)

print("QUESTION")
print(result["question"])

print("\nPROMPT PREVIEW")
print(result["prompt"][:2000])

print("\nSOURCES")
for src in result["sources"]:
    print(src)


## Mengirim prompt ke LLM (Opsional)

Jika ingin menghasilkan jawaban otomatis, kirim `result["prompt"]` ke model pilihan Anda.

Contoh dengan OpenAI SDK:

```python
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-4.1",
    input=result["prompt"]
)

print(response.output_text)
```

Atau gunakan Ollama, LM Studio, atau provider lain dengan mengganti bagian pemanggilan model.


In [ ]:
conn.close()
print("Selesai 🎉")
